# Day 092 — Neural Network Regularization

This notebook covers the Day 92 roadmap requirements using **Python + NumPy only**:
- L1 regularization
- L2 regularization / weight decay
- Dropout mechanism
- Inverted Dropout and probability
- Data augmentation intuition with NumPy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


## 1. L1 Regularization

$$L_{total}=L_{data}+\lambda\sum|w|$$

L1 adds a penalty proportional to the absolute value of each weight.

In [ ]:
weights = np.array([0.5, -0.3, 0.8, -0.2])
data_loss = 0.50
lambda_l1 = 0.01

l1_penalty = lambda_l1 * np.sum(np.abs(weights))
l1_total_loss = data_loss + l1_penalty

print("Weights:", weights)
print("L1 penalty:", l1_penalty)
print("Total loss:", l1_total_loss)


In [ ]:
def l1_regularization(weights, lambda_reg=0.01):
    return lambda_reg * np.sum(np.abs(weights))


print("L1 penalty:", l1_regularization(weights, 0.01))


## 2. L2 Regularization / Weight Decay

$$L_{total}=L_{data}+\lambda\sum w^2$$

L2 penalizes large weights more strongly because the weights are squared.

In [ ]:
lambda_l2 = 0.01

l2_penalty = lambda_l2 * np.sum(weights ** 2)
l2_total_loss = data_loss + l2_penalty

print("Weights:", weights)
print("L2 penalty:", l2_penalty)
print("Total loss:", l2_total_loss)


In [ ]:
def l2_regularization(weights, lambda_reg=0.01):
    return lambda_reg * np.sum(weights ** 2)


print("L2 penalty:", l2_regularization(weights, 0.01))


## 3. L1 vs L2 Penalty


In [ ]:
w = np.linspace(-3, 3, 400)
lambda_reg = 0.2

l1 = lambda_reg * np.abs(w)
l2 = lambda_reg * (w ** 2)

plt.figure(figsize=(8, 5))
plt.plot(w, l1, label="L1: λ|w|")
plt.plot(w, l2, label="L2: λw²")
plt.xlabel("Weight")
plt.ylabel("Penalty")
plt.title("L1 vs L2 Regularization Penalty")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## 4. Dropout Probability

For dropout probability `p`:
- `P(drop) = p`
- `P(keep) = 1 - p`

`p = 0.5` means **each neuron independently has a 50% chance of being dropped**. It does not mean exactly half must be dropped every time.

In [ ]:
x = np.array([10., 20., 30., 40., 50., 60.])
p = 0.5

random_numbers = np.random.rand(len(x))
drop_mask = (random_numbers >= p).astype(float)
output = x * drop_mask

print("Input:", x)
print("Random numbers:", np.round(random_numbers, 3))
print("Mask (1=keep, 0=drop):", drop_mask)
print("After dropout:", output)


## 5. Reusable Dropout Function


In [ ]:
def dropout(x, p=0.5):
    if not 0 <= p < 1:
        raise ValueError("p must be in the range [0, 1).")

    random_numbers = np.random.rand(*x.shape)
    mask = (random_numbers >= p).astype(float)
    return x * mask


for p in [0.2, 0.5, 0.8]:
    print(f"p={p}: {dropout(x, p)}")


## 6. Inverted Dropout

$$y=\frac{x\odot m}{1-p}$$

The surviving activations are scaled by `1 / (1-p)` so their expected value stays approximately unchanged.

In [ ]:
def inverted_dropout(x, p=0.5):
    if not 0 <= p < 1:
        raise ValueError("p must be in the range [0, 1).")

    keep_probability = 1 - p
    random_numbers = np.random.rand(*x.shape)
    mask = (random_numbers >= p).astype(float)
    return (x * mask) / keep_probability


x = np.ones(10)
result = inverted_dropout(x, p=0.5)

print("Input mean:", x.mean())
print("Output:", result)
print("Output mean:", result.mean())


## 7. Data Augmentation with NumPy

We can create slightly modified training samples by adding small noise. This demonstrates the idea of increasing the effective variety of training data.

In [ ]:
def add_noise(X, noise_std=0.10):
    noise = np.random.normal(0, noise_std, size=X.shape)
    return X + noise


X = np.random.randn(100, 2)
X_augmented = add_noise(X, noise_std=0.15)

print("Original shape:", X.shape)
print("Augmented shape:", X_augmented.shape)


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, label="Original")
plt.scatter(X_augmented[:, 0], X_augmented[:, 1], alpha=0.6, label="Augmented")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Simple Data Augmentation with Noise")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## 8. Quick Recap

- **L1:** adds `λ * |w|` penalty.
- **L2:** adds `λ * w²` penalty.
- **Dropout:** each neuron is independently dropped with probability `p` during training.
- **Inverted Dropout:** surviving activations are divided by `1-p` during training.
- **Data Augmentation:** creates more varied training examples from existing data.
